# LeNet-5 Model Implementation

In this notebook, you will:
- Build LeNet-5 for **Handwriting Recognition (Gray Scale Image)** using the TF Keras Functional API

# Load Library

In [2]:
import os
import numpy as np
import h5py
# import matplotlib.pyplot as plt
# import pandas as pd
import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.keras.datasets import mnist # We will remove it later

# Handwriting Recognition (Gray Scale Image)

We'll use Keras' flexible [Functional API](https://www.tensorflow.org/guide/keras/functional) to build a **LeNet-5 Model** that can **recognize** between **0-9 handwriting**.

<img src="images/LeNet-5_Architecture.png">

## 1. Load Data and Split into Train/Test Sets

### 1.1 - Load Data

In [3]:
(train_set_x_orig, train_set_y_orig), (test_set_x_orig, test_set_y_orig) = mnist.load_data()
list_classes = np.unique(train_set_y_orig)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [4]:
# ======= TRAINING SET =======
print("train_set_x_orig shape:", train_set_x_orig.shape)
print("train_set_y_orig shape:", train_set_y_orig.shape)

train_set_x_orig shape: (60000, 28, 28)
train_set_y_orig shape: (60000,)


In [5]:
# ======= TEST SET =======
print("test_set_x_orig shape:", test_set_x_orig.shape)
print("test_set_y_orig shape:", test_set_y_orig.shape)

test_set_x_orig shape: (10000, 28, 28)
test_set_y_orig shape: (10000,)


In [6]:
# ======= CLASSES =======
print("Classes shape:", list_classes.shape)

Classes shape: (10,)


**_Note_**: We can see that:
- **X - feature**: is in **incorrect** format of `(batch_size, height, weight, channels)`. It is lack of `channels`
- **Y - label** is in **incorrect** format. We need `(1, #label)` - a 2D format, not in 1D vector

⇒ We need to convert both X and Y to correct format

In [7]:
# Example of an image from the dataset
index = 7
# plt.imshow(train_set_x_orig[index])
print ("y = " + str(np.squeeze(train_set_y_orig[index])))

y = 3


### 1.2 - Convert X and Y into Correct Format

We need to do that because it helps:
- **Easier for compute Loss**
- **Match with Output model**

**_Note:_** We do **NOT need** to **convert classes** because it does **NOT participate** in the **calculation** nor **input** for the **model**.

#### 1.2.1 - Convert X

We convert from: `(batch_size, height, weight)` => `(batch_size, height, weight, channels)`

In [8]:
# ======= Convert X =======
img_height, img_width = train_set_x_orig.shape[1], train_set_x_orig.shape[2]

train_set_x_orig = train_set_x_orig.reshape(-1, img_height, img_width, 1)
test_set_x_orig = test_set_x_orig.reshape(-1, img_height, img_width, 1)
print("train_set_x_orig shape:", train_set_x_orig.shape)
print("test_set_x_orig shape:", test_set_x_orig.shape)

train_set_x_orig shape: (60000, 28, 28, 1)
test_set_x_orig shape: (10000, 28, 28, 1)


#### 1.2.2 - Convert Y

We convert from: 1D `(#label,)` => 2D `(1, #label)`

In [9]:
# ======= Convert Y =======
train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))
print("train_set_y_orig shape:", train_set_y_orig.shape)
print("test_set_y_orig shape:", test_set_y_orig.shape)

train_set_y_orig shape: (1, 60000)
test_set_y_orig shape: (1, 10000)


### 1.3 - Convert Classes to One-Hot Representation

We need to know what the representation of classes is to know how we can convert it to One-hot representation.

In [10]:
print("Train: y = " + str(np.squeeze(train_set_y_orig)) + ", Size: " + str(train_set_y_orig.shape))

Train: y = [5 0 4 ... 5 6 8], Size: (1, 60000)


In [11]:
def convert_to_one_hot(Y, C):
    Y = np.eye(C)[Y.reshape(-1)].T
    return Y

### 1.4 - Split Data

Images are **28x28** pixels in Gray format (1 channels):
- **X_train/X_test shape**: `(batch_size, height, width, channels)`
- **Y_train/Y_test shape**: `(batch_size, #classes_one_hot_represent)`
- **Classes shape**: `(#classes,)`

In [12]:
# Normalize image vectors
X_train = train_set_x_orig/255.
X_test = test_set_x_orig/255.

# Reshape + Convert Y to One-hot
Y_train = convert_to_one_hot(train_set_y_orig, list_classes.shape[0]).T
Y_test = convert_to_one_hot(test_set_y_orig, list_classes.shape[0]).T

# Classes
classes = list_classes

print ("Number of training examples = " + str(X_train.shape[0]))
print ("Number of test examples = " + str(X_test.shape[0]))
print ("X_train shape: " + str(X_train.shape))
print ("Y_train shape: " + str(Y_train.shape))
print ("X_test shape: " + str(X_test.shape))
print ("Y_test shape: " + str(Y_test.shape))
print ("Classes shape: " + str(classes.shape))

Number of training examples = 60000
Number of test examples = 10000
X_train shape: (60000, 28, 28, 1)
Y_train shape: (60000, 10)
X_test shape: (10000, 28, 28, 1)
Y_test shape: (10000, 10)
Classes shape: (10,)


## 2 - Model Processing

### 2.1 - Create LeNet-5 Model Pipeline

<img src="images/LeNet-5_Pipeline.png">

Use the functions above!

Also, plug in the following parameters for all the steps:

 - [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D): Use 6 5 by 5 filters, stride 1, padding is "VALID"
 - [Sigmoid](https://www.tensorflow.org/api_docs/python/tf/keras/activations/sigmoid)
 - [AvgPool2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/AvgPool2D): Use an 2 by 2 filter size and an 2 by 2 stride, padding is "VALID"
 - **Sigmoid**
 - **Conv2D**: Use 16 5 by 5 filters, stride 1, padding is "VALID"
 - **Sigmoid**
 - **AvgPool2D**: Use an 2 by 2 filter size and an 2 by 2 stride, padding is "VALID"
 - **Sigmoid**
 - [Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten) the previous output.
 - Fully-connected ([Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)) layer: Apply a fully connected layer with 120 neurons.
 - **Sigmoid**
 - **Dense**: Apply a fully connected layer with 84 neurons.
 - **Sigmoid**
 - **Dense**: Apply a fully connected layer with 10 neurons.
 - [Softmax](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Softmax): Output probabilities for 10 classes.

**_Note_**: With LeNet-5, we use Input Image (32x32x1). So, we need to **Padding Original Dataset** from **(28x28x1) ⇒ (32x32x1)**

In [13]:
def LeNet5(input_shape):
    """
    Implements the forward propagation for the LeNet-5 model

    Arguments:
    input_img -- input dataset, of shape (input_shape) - (28x28x1)

    Returns:
    model -- TF Keras model (object containing the information for the entire training process)
    """
    input_img = tf.keras.Input(shape=input_shape)
    A0 = tfl.ZeroPadding2D(padding=(2,2))(input_img)

    # 1. Conv
    Z1 = tfl.Conv2D(filters=6, kernel_size=(5, 5), strides=(1, 1), padding='valid')(A0)

    # 2. Sigmoid
    A1 = tfl.Activation('sigmoid')(Z1)

    # 3. AvgPool
    A1_pool = tfl.AvgPool2D(pool_size=(2, 2), strides=(2, 2), padding='valid')(A1)

    # 4. Sigmoid
    A1_pool_sig = tfl.Activation('sigmoid')(A1_pool)

    # 5. Conv
    Z2 = tfl.Conv2D(filters=16, kernel_size=(5, 5), strides=(1, 1), padding='valid')(A1_pool_sig)

    # 6. Sigmoid
    A2 = tfl.Activation('sigmoid')(Z2)

    # 7. AvgPool
    A2_pool = tfl.AvgPool2D(pool_size=(2, 2), strides=(2, 2), padding='valid')(A2)

    # 8. Sigmoid
    A2_pool_sig = tfl.Activation('sigmoid')(A2_pool)

    # 9. Flatten
    F = tfl.Flatten()(A2_pool_sig)

    # 10. Dense 120
    Z3 = tfl.Dense(units=120)(F)

    # 11. Sigmoid
    A3 = tfl.Activation('sigmoid')(Z3)

    # 12. Dense 84
    Z4 = tfl.Dense(units=84)(A3)

    # 13. Sigmoid
    A4 = tfl.Activation('sigmoid')(Z4)

    # 14. Dense 10
    Z5 = tfl.Dense(units=10)(A4)

    # 15. Output: Softmax
    outputs = tfl.Softmax(axis=-1)(Z5)

    model = tf.keras.Model(inputs=input_img, outputs=outputs)
    return model

In [14]:
LeNet5_model = LeNet5((28, 28, 1))
LeNet5_model.summary()

W0000 00:00:1774985440.200511    1867 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1774985440.675981    1867 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5233 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d (ZeroPadding2D)  │ (None, 32, 32, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 28, 28, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 10, 10, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 84)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### 2.2 - Set up before Training

Before Training, we need to define:
- How to **Calculate Error**
- How to **Update Weight**
- How to **Measure Performance**

In [15]:
LeNet5_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

### 2.3 - Train Model


In [16]:
save_dir = "models/lenet5_run"
os.makedirs(save_dir, exist_ok=True)

In [17]:
# Checkpoint: Save best model
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(save_dir, "best_lenet5.keras"),
    monitor="val_loss",
    save_best_only=True
)

In [18]:
# Training Process
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train)).batch(64)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, Y_test)).batch(64)
history = LeNet5_model.fit(
    train_dataset,
    epochs=100,
    validation_data=test_dataset,
    callbacks=[checkpoint]
)

Epoch 1/100


I0000 00:00:1774985450.302514    1943 service.cc:153] XLA service 0x707af00055f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774985450.302574    1943 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5060 Laptop GPU, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1774985450.382783    1943 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774985450.749841    1943 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1774985450.777080    1943 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1955__.27


 24/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1093 - loss: 2.4091

I0000 00:00:1774985462.417345    1943 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


934/938 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1075 - loss: 2.3124

I0000 00:00:1774985468.581641    1945 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1955__.27


938/938 ━━━━━━━━━━━━━━━━━━━━ 34s 19ms/step - accuracy: 0.1065 - loss: 2.3071 - val_accuracy: 0.1390 - val_loss: 2.3035
Epoch 2/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.3248 - loss: 1.8884 - val_accuracy: 0.6740 - val_loss: 0.9829
Epoch 3/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.7345 - loss: 0.8003 - val_accuracy: 0.7839 - val_loss: 0.6747
Epoch 4/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8106 - loss: 0.5756 - val_accuracy: 0.8354 - val_loss: 0.4897
Epoch 5/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8394 - loss: 0.4889 - val_accuracy: 0.8584 - val_loss: 0.4288
Epoch 6/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.8569 - loss: 0.4386 - val_accuracy: 0.8666 - val_loss: 0.4007
Epoch 7/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8698 - loss: 0.4025 - val_accuracy: 0.8750 - val_loss: 0.3774
Epoch 8/100
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8792 - loss: 0.3741 - val_accuracy: 0.88

In [19]:
# Save Final Model
LeNet5_model.save(os.path.join(save_dir, "final_lenet5.keras"))